# KEGG pathway ORA on FDR-filtered logFCs

For each (perturbation, drug-context) pair we have a posterior-mean logFC vector across
~18k response genes plus a matching FDR vector.  We:

1. Mask logFCs whose FDR exceeds `FDR_THRESHOLD` (default `0.1`).
2. For each perturbation, build three significant-gene sets: **combined**, **up**
   (`logFC > 0` & `FDR < t`) and **down** (`logFC < 0` & `FDR < t`).
3. Run **hypergeometric ORA** of each set against the KEGG pathway library
   (`MSigDB c2.cp.kegg`, ~186 pathways).
4. BH-correct p-values **within each (perturbation, direction)** across pathways.
5. Save long-format results per drug.

First pass runs only `DMSO_round2` end-to-end.  Once we are happy with timing
and sanity, the same loop extends to all 16 contexts.

In [13]:
2**0.15

1.109569472067845

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

## Paths and parameters

In [2]:
PROJECT_DIR = Path('/home/beraslan/Projects/ChemoGeneticScreens')
PM_DIR      = PROJECT_DIR / 'PosteriorMeanMatrices'
FDR_DIR     = PROJECT_DIR / 'FDR_matrices'
OUT_DIR     = PROJECT_DIR / 'PathwayORA' / 'KEGG'
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEGG_GMTS = [
    Path('/home/beraslan/Projects/ModuleFinder/MuVI/msigdb/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt'),
    Path('/home/beraslan/Projects/ModuleFinder/MuVI/msigdb/c2.cp.kegg_medicus.v2024.1.Hs.symbols.gmt'),
]

FDR_THRESHOLD     = 0.01   # genes with FDR < this are 'significant'
MIN_PATHWAY_SIZE  = 10     # drop tiny pathways (after intersecting with universe)
MAX_PATHWAY_SIZE  = 500    # drop huge pathways
MIN_SIG_PER_PERT  = 5      # skip perturbations with too few sig genes (ORA uninformative)

FIRST_DRUG = 'DMSO_round2'  # proof-of-concept context

## Load KEGG pathways from MSigDB GMT

In [3]:
def parse_gmt(path):
    """Return {pathway_name: set(genes)} from a tab-separated GMT."""
    out = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            name, _url, *genes = parts
            genes = {g for g in genes if g}
            if genes:
                out[name] = genes
    return out


kegg_raw = {}
for p in KEGG_GMTS:
    sub = parse_gmt(p)
    print(f"  {p.name}: {len(sub)} pathways")
    kegg_raw.update(sub)
print(f"Loaded {len(kegg_raw)} KEGG pathways (union of {len(KEGG_GMTS)} GMTs)")
print(f"Sizes (raw): min={min(len(v) for v in kegg_raw.values())}  "
      f"median={int(np.median([len(v) for v in kegg_raw.values()]))}  "
      f"max={max(len(v) for v in kegg_raw.values())}")

  c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt: 186 pathways
  c2.cp.kegg_medicus.v2024.1.Hs.symbols.gmt: 658 pathways
Loaded 844 KEGG pathways (union of 2 GMTs)
Sizes (raw): min=5  median=15  max=389


## Load PosteriorMean and FDR matrices for the first drug

Both files have rows = perturbations, columns = response genes.  We intersect
perturbation rows (PosteriorMean has 2,212; FDR has 2,367) and assume the gene
columns match (they do — verified at the shell).

In [4]:
def load_drug(drug):
    """Load aligned (logFC, FDR) DataFrames indexed by perturbation, columns = genes."""
    pm  = pd.read_csv(PM_DIR  / f'PosteriorMean_matrix_{drug}.csv', index_col=0)
    fdr = pd.read_csv(FDR_DIR / f'{drug}_FDRs.csv',                index_col=0)
    common_perts = pm.index.intersection(fdr.index)
    common_genes = pm.columns.intersection(fdr.columns)
    pm  = pm.loc[common_perts,  common_genes]
    fdr = fdr.loc[common_perts, common_genes]
    return pm, fdr

logfc_df, fdr_df = load_drug(FIRST_DRUG)
print(f"{FIRST_DRUG}: logFC {logfc_df.shape}, FDR {fdr_df.shape}")
print(f"perturbations: {logfc_df.shape[0]}, response genes: {logfc_df.shape[1]}")
print(f"% of cells with FDR < {FDR_THRESHOLD}: {(fdr_df.values < FDR_THRESHOLD).mean()*100:.2f}%")

DMSO_round2: logFC (2212, 18154), FDR (2212, 18154)
perturbations: 2212, response genes: 18154
% of cells with FDR < 0.01: 1.54%


## Build the gene universe and the membership matrix

**Universe** = response genes that are in *any* KEGG pathway.  This is
the standard ORA convention: tested genes that could in principle map to
the pathway library.  Pathway sizes (`K`) and per-perturbation sig counts
(`n`) are recomputed on this universe, not on the raw 18k columns.

In [5]:
tested_genes = list(logfc_df.columns)
kegg_union   = set().union(*kegg_raw.values())
universe     = [g for g in tested_genes if g in kegg_union]
gene_to_idx  = {g: i for i, g in enumerate(universe)}
N_UNIV       = len(universe)

# Filter pathways to the universe and by size
kegg = {}
for name, genes in kegg_raw.items():
    g_in_univ = genes & set(universe)
    if MIN_PATHWAY_SIZE <= len(g_in_univ) <= MAX_PATHWAY_SIZE:
        kegg[name] = g_in_univ
pathway_names = sorted(kegg)
P = len(pathway_names)
print(f"Universe (tested ∩ KEGG): {N_UNIV} genes")
print(f"Pathways kept after size filter [{MIN_PATHWAY_SIZE}, {MAX_PATHWAY_SIZE}]: {P}")

# Membership matrix M: (P, N_UNIV) boolean
M = np.zeros((P, N_UNIV), dtype=bool)
for i, name in enumerate(pathway_names):
    idx = [gene_to_idx[g] for g in kegg[name]]
    M[i, idx] = True
K_sizes = M.sum(axis=1)            # (P,)  pathway sizes in universe
print(f"Pathway sizes — min={K_sizes.min()}, median={int(np.median(K_sizes))}, max={K_sizes.max()}")

Universe (tested ∩ KEGG): 5658 genes
Pathways kept after size filter [10, 500]: 568
Pathway sizes — min=10, median=21, max=352


## Build the per-perturbation significance matrices

Three boolean matrices `(n_perturbations × N_UNIV)`:

- `S_any`  : `FDR < t`
- `S_up`   : `FDR < t & logFC > 0`
- `S_down` : `FDR < t & logFC < 0`

All three restricted to the universe columns (drop genes outside KEGG).

In [6]:
lfc_u = logfc_df[universe].values        # (n_perts, N_UNIV)
fdr_u = fdr_df[universe].values

sig_any  = fdr_u < FDR_THRESHOLD
sig_up   = sig_any & (lfc_u > 0)
sig_down = sig_any & (lfc_u < 0)

perts = list(logfc_df.index)
n_per_pert_any  = sig_any.sum(axis=1)
n_per_pert_up   = sig_up.sum(axis=1)
n_per_pert_down = sig_down.sum(axis=1)

summary = pd.DataFrame({
    'perturbation': perts,
    'n_sig_any':  n_per_pert_any,
    'n_sig_up':   n_per_pert_up,
    'n_sig_down': n_per_pert_down,
})
print(summary.describe()[['n_sig_any', 'n_sig_up', 'n_sig_down']].round(1).to_string())
print(f"\nperturbations with >= {MIN_SIG_PER_PERT} sig genes (any direction): "
      f"{(n_per_pert_any >= MIN_SIG_PER_PERT).sum()} / {len(perts)}")

       n_sig_any  n_sig_up  n_sig_down
count     2212.0    2212.0      2212.0
mean       102.8      58.2        44.6
std        201.0     103.1       102.1
min          5.0       3.0         0.0
25%         18.0      15.0         2.0
50%         28.0      21.0         6.0
75%         86.0      54.0        34.0
max       2061.0    1176.0       934.0

perturbations with >= 5 sig genes (any direction): 2212 / 2212


## Vectorised hypergeometric ORA

For each (perturbation `p`, pathway `i`):

- `N`  = universe size
- `K_i` = pathway size in universe
- `n_p` = number of sig genes for perturbation `p`
- `k_pi` = sig genes that fall in pathway `i`

p-value = `P(X >= k_pi) = hypergeom.sf(k_pi - 1, N, K_i, n_p)`.

We compute the full `(n_perts × P)` matrix at once via `S @ M.T`, then call
`hypergeom.sf` on the whole array.  BH correction is applied **within each
perturbation row** (across pathways).

In [7]:
def run_ora(S, M, K_sizes, N, perts, pathway_names, direction, min_sig=5):
    """Vectorised hypergeometric ORA.

    Parameters
    ----------
    S : (n_perts, N) bool — significance matrix for one direction
    M : (P, N) bool — pathway membership
    K_sizes : (P,) int — pathway sizes in universe
    N : int — universe size
    perts, pathway_names : labels
    direction : str — label for output
    min_sig : skip perturbations with fewer than this many sig genes

    Returns
    -------
    long-format DataFrame with one row per (perturbation, pathway).
    """
    n_p = S.sum(axis=1)                              # (n_perts,)
    keep = n_p >= min_sig
    if not keep.any():
        return pd.DataFrame(columns=[
            'perturbation', 'pathway', 'direction',
            'n_sig', 'k_overlap', 'K_size',
            'p_value', 'q_bh', 'odds_ratio',
        ])

    S_kept = S[keep]
    perts_kept = [perts[i] for i, k in enumerate(keep) if k]
    n_kept = n_p[keep]                               # (m,)
    k_mat  = S_kept.astype(np.int32) @ M.astype(np.int32).T   # (m, P)

    # Broadcast inputs to (m, P) for hypergeom.sf
    K_b = np.broadcast_to(K_sizes,        k_mat.shape)
    n_b = np.broadcast_to(n_kept[:, None], k_mat.shape)
    p_mat = hypergeom.sf(k_mat - 1, N, K_b, n_b)

    # BH within perturbation (across pathways)
    q_mat = np.empty_like(p_mat)
    for i in range(p_mat.shape[0]):
        _, q, _, _ = multipletests(p_mat[i], method='fdr_bh')
        q_mat[i] = q

    expected = (K_b * n_b) / N
    odds = np.where(expected > 0, k_mat / expected, np.nan)

    rows = []
    for i, p_lbl in enumerate(perts_kept):
        for j, path in enumerate(pathway_names):
            rows.append((
                p_lbl, path, direction,
                int(n_kept[i]), int(k_mat[i, j]), int(K_sizes[j]),
                float(p_mat[i, j]), float(q_mat[i, j]), float(odds[i, j]),
            ))
    return pd.DataFrame(rows, columns=[
        'perturbation', 'pathway', 'direction',
        'n_sig', 'k_overlap', 'K_size',
        'p_value', 'q_bh', 'odds_ratio',
    ])

## Run ORA for the three directions

In [8]:
%time res_any  = run_ora(sig_any,  M, K_sizes, N_UNIV, perts, pathway_names, 'any',  min_sig=MIN_SIG_PER_PERT)
%time res_up   = run_ora(sig_up,   M, K_sizes, N_UNIV, perts, pathway_names, 'up',   min_sig=MIN_SIG_PER_PERT)
%time res_down = run_ora(sig_down, M, K_sizes, N_UNIV, perts, pathway_names, 'down', min_sig=MIN_SIG_PER_PERT)

results = pd.concat([res_any, res_up, res_down], ignore_index=True)
print(f"\nRows in long-format result: {len(results):,}")
print(results.head())

CPU times: user 12.3 s, sys: 204 ms, total: 12.5 s
Wall time: 12.5 s
CPU times: user 10.1 s, sys: 120 ms, total: 10.2 s
Wall time: 10.2 s
CPU times: user 6.67 s, sys: 75.8 ms, total: 6.75 s
Wall time: 6.75 s

Rows in long-format result: 3,207,496
  perturbation                                          pathway direction  \
0         A1BG                            KEGG_ABC_TRANSPORTERS       any   
1         A1BG                      KEGG_ACUTE_MYELOID_LEUKEMIA       any   
2         A1BG                           KEGG_ADHERENS_JUNCTION       any   
3         A1BG             KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY       any   
4         A1BG  KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM       any   

   n_sig  k_overlap  K_size  p_value  q_bh  odds_ratio  
0     13          0      42      1.0   1.0         0.0  
1     13          0      55      1.0   1.0         0.0  
2     13          0      72      1.0   1.0         0.0  
3     13          0      67      1.0   1.0         0.0  
4    

CPU times: user 5.45 s, sys: 39.7 ms, total: 5.49 s
Wall time: 5.49 s


CPU times: user 4.14 s, sys: 3.83 ms, total: 4.14 s
Wall time: 4.14 s

Rows in long-format result: 1,170,976
  perturbation                                          pathway direction  \
0         A1BG                            KEGG_ABC_TRANSPORTERS       any   
1         A1BG                      KEGG_ACUTE_MYELOID_LEUKEMIA       any   
2         A1BG                           KEGG_ADHERENS_JUNCTION       any   
3         A1BG             KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY       any   
4         A1BG  KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM       any   

   n_sig  k_overlap  K_size   p_value  q_bh  odds_ratio  
0     36          0      42  1.000000   1.0    0.000000  
1     36          0      55  1.000000   1.0    0.000000  
2     36          1      72  0.415009   1.0    1.885802  
3     36          0      67  1.000000   1.0    0.000000  
4     36          0      30  1.000000   1.0    0.000000  


## Sanity check — top hits for a few example perturbations

Pick a few perturbations and look at the top KEGG pathways they enrich (any
direction).  We do not have a hard-coded ground truth here, but for KOs of
ribosomal genes, splicing factors, etc., the corresponding KEGG pathway
should rank near the top.

In [9]:
# Pick the 3 perturbations with the most sig genes — they should give the cleanest signal
top_perts = (summary.sort_values('n_sig_any', ascending=False)
             .head(3)['perturbation'].tolist())

for p in top_perts:
    sub = (results[(results.perturbation == p) & (results.direction == 'any')]
           .sort_values('q_bh').head(8)
           [['pathway', 'k_overlap', 'K_size', 'n_sig', 'p_value', 'q_bh', 'odds_ratio']])
    print(f"\n=== {p}  (n_sig_any = {summary.set_index('perturbation').loc[p, 'n_sig_any']}) ===")
    print(sub.to_string(index=False))


=== STRAP  (n_sig_any = 2061) ===
                    pathway  k_overlap  K_size  n_sig  p_value     q_bh  odds_ratio
     KEGG_ADHERENS_JUNCTION         42      72   2061 0.000116 0.023307    1.601407
    KEGG_ALZHEIMERS_DISEASE         81     159   2061 0.000104 0.023307    1.398533
    KEGG_PARKINSONS_DISEASE         65     123   2061 0.000126 0.023307    1.450752
   KEGG_HUNTINGTONS_DISEASE         85     170   2061 0.000164 0.023307    1.372635
KEGG_ACUTE_MYELOID_LEUKEMIA         33      55   2061 0.000300 0.034063    1.647162
 KEGG_P53_SIGNALING_PATHWAY         38      68   2061 0.000784 0.063616    1.534121
  KEGG_RENAL_CELL_CARCINOMA         38      68   2061 0.000784 0.063616    1.534121
         KEGG_AXON_GUIDANCE         63     126   2061 0.001115 0.070389    1.372635

=== EFEMP1  (n_sig_any = 1864) ===
                                         pathway  k_overlap  K_size  n_sig  p_value     q_bh  odds_ratio
                         KEGG_PATHWAYS_IN_CANCER        142     322 


=== EFEMP1  (n_sig_any = 1952) ===
                       pathway  k_overlap  K_size  n_sig  p_value     q_bh  odds_ratio
               KEGG_CELL_CYCLE         74     124   1952 0.000005 0.001006    1.494381
       KEGG_PARKINSONS_DISEASE         71     123   1952 0.000041 0.003783    1.445455
      KEGG_HUNTINGTONS_DISEASE         92     170   1952 0.000099 0.004550    1.355159
       KEGG_PATHWAYS_IN_CANCER        161     322   1952 0.000098 0.004550    1.252049
KEGG_OXIDATIVE_PHOSPHORYLATION         70     126   1952 0.000238 0.007407    1.391166
           KEGG_FOCAL_ADHESION        102     195   1952 0.000242 0.007407    1.309836
           KEGG_TIGHT_JUNCTION         71     129   1952 0.000314 0.008241    1.378225
KEGG_INSULIN_SIGNALING_PATHWAY         73     134   1952 0.000392 0.009017    1.364173

=== CSDE1  (n_sig_any = 1884) ===
                     pathway  k_overlap  K_size  n_sig  p_value     q_bh  odds_ratio
      KEGG_ADHERENS_JUNCTION         46      72   1884 0.0000

## Save

Long-format Parquet (one row per (perturbation, pathway, direction))
for easy filtering.  Wide-format `−log10(q_bh)` matrices can be derived
with `pivot_table` downstream.

In [ ]:
out_path = OUT_DIR / f'KEGG_ORA_{FIRST_DRUG}.parquet'
results.to_parquet(out_path, index=False)
print(f"Saved {out_path}  ({len(results):,} rows)")

# Tiny CSV summary too — easy to eyeball
summary_out = OUT_DIR / f'KEGG_ORA_{FIRST_DRUG}_summary.csv'
summary.to_csv(summary_out, index=False)
print(f"Saved {summary_out}")